# 009 Groundedness Check

上一课已经能生成带引用的回答，但“带引用”不等于“事实被证据支持”。

这一课学习 `groundedness check`：判断回答里的声明是否真的能从证据中推出。

你要掌握三件事：

1. 格式验证、引用验证、事实一致性验证不是一回事。
2. 如何把回答拆成可检查的 claim。
3. 如何用规则 + 小模型评审做一版最低可用的 groundedness 检查。


## 1. 为什么第 008 课还不够

第 008 课验证的是：

```text
回答非空？
有没有引用 [T1] / [G1]？
引用编号是否存在？
```

这相当于 Java 里做 DTO 字段校验：字段存在、格式正确、枚举值合法。

但业务正确性还没验证。

例如模型回答里写了“演练与培训”“监督与评价”，即使它后面带了 `[T1]`，也不代表 `[T1]` 真的支持这些说法。

所以这一课要加一层更接近业务语义的验证：

```text
claim -> cited evidence -> supported / unsupported / unclear
```


## 2. 导入依赖

本课读取第 008 课生成的 `qa_result.json` 和第 007 课生成的 `evidence_package.json`。


In [1]:
import importlib.metadata
import json
import os
import re
from hashlib import sha1
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

print('openai', importlib.metadata.version('openai'))

openai 2.36.0


## 3. 读取问答结果和证据包

如果你还没有执行第 008 课，请先执行 `008-qa-with-evidence-and-validation.ipynb`。


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
evidence_package_path = generated_dir / 'evidence_package.json'
qa_result_path = generated_dir / 'qa_result.json'
groundedness_report_path = generated_dir / 'groundedness_report.json'

if not evidence_package_path.exists():
    raise FileNotFoundError(f'请先执行第 007 课: {evidence_package_path}')
if not qa_result_path.exists():
    raise FileNotFoundError(f'请先执行第 008 课: {qa_result_path}')

evidence_package = json.loads(evidence_package_path.read_text(encoding='utf-8'))
qa_result = json.loads(qa_result_path.read_text(encoding='utf-8'))
answer = qa_result['answer']

print('doc_id:', doc_id)
print('question:', qa_result['question'])
print('answer_source:', qa_result.get('answer_source'))
print('answer_chars:', len(answer))
print('text_evidence_count:', len(evidence_package.get('text_evidence', [])))
print('graph_evidence_count:', len(evidence_package.get('graph_evidence', [])))

doc_id: 63b7d4d0675426b5
question: 密引水渠引水流量不能满足泄洪要求应该怎么处理？
answer_source: model
answer_chars: 698
text_evidence_count: 6
graph_evidence_count: 10


## 4. 重建 citation map

第 008 课里 `[T1]`、`[G1]` 是临时编号。

本课需要重建编号到证据的映射，才能检查一个 claim 引用的证据到底是什么。


In [3]:
def normalize_space(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def build_citation_map(package: dict, text_limit: int = 6, graph_limit: int = 6) -> dict:
    citation_map = {}
    for index, item in enumerate(package.get('text_evidence', [])[:text_limit], start=1):
        citation_map[f'T{index}'] = {
            'type': 'text',
            'chunk_id': item.get('chunk_id'),
            'page_start': item.get('page_start'),
            'page_end': item.get('page_end'),
            'evidence_text': normalize_space(item.get('text', '')),
        }
    for index, item in enumerate(package.get('graph_evidence', [])[:graph_limit], start=1):
        citation_map[f'G{index}'] = {
            'type': 'graph',
            'chunk_id': item.get('chunk_id'),
            'page_start': item.get('page_start'),
            'page_end': item.get('page_end'),
            'subject': item.get('subject'),
            'predicate': item.get('predicate'),
            'object': item.get('object'),
            'evidence_text': normalize_space(
                f"{item.get('subject')} {item.get('predicate')} {item.get('object')} {item.get('evidence', '')}"
            ),
        }
    return citation_map

citation_map = build_citation_map(evidence_package)
print('citations:', sorted(citation_map))
preview_id = 'T1' if 'T1' in citation_map else (sorted(citation_map)[0] if citation_map else None)
if preview_id is None:
    raise ValueError('当前 evidence_package 没有任何可用证据，请先重新执行第 007 课')
print(f'{preview_id} preview:', citation_map[preview_id]['evidence_text'][:300])

citations: ['G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6']
T1 preview: 图7 预报调度2（最大下泄流量3000 m3/s）过程示例 表 7 不同调度措施的调洪成果 预泄流 入库洪 最高水 最大出库流 最高库 超汛限 第10 调度模 量 峰 位出现 削峰率 量 水位 水位时 日水位 式 （m³/s （m³/s 时刻 （%） （m³/s） （m） 间（d） (m) ） ） （h） 规程调 200 17300 15400 157.46 10 62 153.74 11.0 度 预报调 200 17300 4000 157.50 9 88 155.38 76.9 度 200 17300 3000 157.72 9 88 155.74 82.7 4.3.4 超标准洪水调度 当


## 5. 把回答拆成 claim

业务上不要直接评估整段回答。

更合理的方式是先拆成多个 claim：

```text
claim_id: C1
text: 方案包含编制目的、编制依据、适用范围
citations: [T3]
```

这样后续才能逐条判断 supported / unsupported。


In [4]:
def extract_citations(text: str) -> list[str]:
    return re.findall(r'\[([TG]\d+)\]', text)


def strip_markdown_prefix(line: str) -> str:
    line = line.strip()
    line = re.sub(r'^#{1,6}\s*', '', line)
    line = re.sub(r'^[-*]\s*', '', line)
    line = re.sub(r'^\d+[.)、]\s*', '', line)
    return line.strip()


def split_answer_into_claims(answer: str) -> list[dict]:
    claims = []
    for raw_line in answer.splitlines():
        line = strip_markdown_prefix(raw_line)
        if not line:
            continue
        if line.startswith('提供的要点') or line.startswith('文本证据来源'):
            continue
        if len(line) < 8:
            continue
        # 跳过大段原文证据转述，避免把引用材料再次当成模型 claim。
        if 'chunk=' in line or '页码' in line:
            continue
        citations = extract_citations(line)
        clean_text = re.sub(r'\[[TG]\d+\]', '', line).strip(' ：:；;')
        if clean_text:
            claims.append({
                'claim_id': f'C{len(claims) + 1}',
                'text': clean_text,
                'citations': citations,
            })
    return claims

claims = split_answer_into_claims(answer)
print('claim_count:', len(claims))
pprint(claims[:12])

claim_count: 7
[{'citations': [],
  'claim_id': 'C1',
  'text': '根据提供的证据，关于“密引水渠引水流量不能满足泄洪要求”的处理方式，证据中并未直接给出针对该特定情况的独立处理措施，但提供了相关的调度原则和超标准洪水下的应急方案。以下是基于证据的要点'},
 {'citations': ['T2'],
  'claim_id': 'C2',
  'text': '**常规控泄流量包含引水流量**：在正常调度规程中，水库的控泄流量是包含京密引水渠引水流量的。例如，当水库水位在152.00m至154.32m之间时，控泄流量为600m³/s，其中已包含京密引水渠引水流量50m³/s '
          '。'},
 {'citations': ['T2'],
  'claim_id': 'C3',
  'text': '**高水位时全开闸门不控泄**：当水库水位超过157.50m时，输、泄水建筑物闸门全部开启，不再进行控泄，此时引水流量对泄洪能力的限制不再适用 '
          '。'},
 {'citations': ['T1'],
  'claim_id': 'C4',
  'text': '**超标准洪水下的极端措施**：若发生超标准洪水，且敞开各输、泄水建筑物泄洪仍不能满足工程安全要求时，经北京市水务局批准后，可选择炸开副坝的形式进行泄洪 '
          '。'},
 {'citations': ['T1'],
  'claim_id': 'C5',
  'text': '**利用预报调度预泄**：通过实施预报调度，根据洪水预报结果提前预泄，以降低库水位，从而为后续泄洪腾出库容。例如在预报调度模式下，最大下泄流量可达3000m³/s或4000m³/s，远高于常规调度 '
          '。'},
 {'citations': ['G1', 'G1'],
  'claim_id': 'C6',
  'text': '**证据中未明确具体替代方案**：虽然图谱证据提到了“京密引水渠引水流量不能满足泄洪要求”这一情境，但并未提供具体的处理步骤或替代调度方案，仅作为背景信息存在 '
          '。'},
 {'citations': ['T1', 'T6', 'G1', 'G

## 6. 规则版 groundedness 检查

第一版用规则，不调用模型。

规则很简单：

1. claim 没有引用，记为 `missing_citation`。
2. 引用了不存在的证据，记为 `unknown_citation`。
3. claim 的关键词和引用证据重合很少，记为 `weak_overlap`。
4. 否则记为 `overlap_supported`。

它不够智能，但有两个优点：便宜、可解释。


In [5]:
STOPWORDS = {
    '主要', '内容', '方案', '密云', '水库', '防御', '洪水', '涉及', '包括', '以及',
    '进行', '提供', '详细', '强调', '提到', '明确', '规定', '说明', '重要性',
}


def tokenize_for_check(text: str) -> set[str]:
    raw_tokens = re.findall(r'[\u4e00-\u9fff]{2,}|[a-zA-Z0-9_]+', text)
    tokens = set()
    for raw in raw_tokens:
        raw = raw.lower()
        if re.fullmatch(r'[\u4e00-\u9fff]+', raw):
            if len(raw) <= 4 and raw not in STOPWORDS:
                tokens.add(raw)
            for size in [2, 3, 4]:
                for index in range(0, max(len(raw) - size + 1, 0)):
                    token = raw[index:index + size]
                    if token not in STOPWORDS:
                        tokens.add(token)
        elif raw not in STOPWORDS:
            tokens.add(raw)
    return tokens


def extract_hard_terms(text: str) -> list[str]:
    """抽取很难靠近义词替代的约束词。

    例如“每季度”“全员演练”这类词如果 claim 里出现，证据里也应该直接出现或强相关。
    这不是完整 NLI，只是为了避免纯关键词重合把明显坏 claim 判成 supported。
    """
    patterns = [r'每季度', r'每月', r'每年', r'全员', r'演练', r'\\d+\\s*次', r'\\d+\\s*小时', r'\\d+\\s*天']
    terms = []
    for pattern in patterns:
        terms.extend(re.findall(pattern, text))
    return sorted(set(terms))


def rule_check_claim(claim: dict, citation_map: dict, min_overlap: int = 2) -> dict:
    citations = claim.get('citations', [])
    if not citations:
        return {**claim, 'status': 'missing_citation', 'overlap_terms': [], 'reason': 'claim 没有引用证据编号'}

    unknown = [citation for citation in citations if citation not in citation_map]
    if unknown:
        return {**claim, 'status': 'unknown_citation', 'overlap_terms': [], 'reason': f'未知引用: {unknown}'}

    claim_terms = tokenize_for_check(claim['text'])
    evidence_text = ' '.join(citation_map[citation]['evidence_text'] for citation in citations)
    hard_terms = extract_hard_terms(claim['text'])
    missing_hard_terms = [term for term in hard_terms if term not in evidence_text]
    if missing_hard_terms:
        return {
            **claim,
            'status': 'missing_required_terms',
            'overlap_terms': [],
            'missing_required_terms': missing_hard_terms,
            'reason': f'claim 含有精确约束词，但引用证据未出现: {missing_hard_terms}',
        }

    evidence_terms = tokenize_for_check(evidence_text)
    overlap_terms = sorted(claim_terms & evidence_terms)

    if len(overlap_terms) >= min_overlap:
        status = 'overlap_supported'
        reason = f'claim 和引用证据有 {len(overlap_terms)} 个关键词重合'
    else:
        status = 'weak_overlap'
        reason = f'关键词重合不足，只有 {len(overlap_terms)} 个'

    return {
        **claim,
        'status': status,
        'overlap_terms': overlap_terms[:20],
        'missing_required_terms': [],
        'reason': reason,
    }

rule_results = [rule_check_claim(claim, citation_map) for claim in claims]
pprint(rule_results[:12])

[{'citations': [],
  'claim_id': 'C1',
  'overlap_terms': [],
  'reason': 'claim 没有引用证据编号',
  'status': 'missing_citation',
  'text': '根据提供的证据，关于“密引水渠引水流量不能满足泄洪要求”的处理方式，证据中并未直接给出针对该特定情况的独立处理措施，但提供了相关的调度原则和超标准洪水下的应急方案。以下是基于证据的要点'},
 {'citations': ['T2'],
  'claim_id': 'C2',
  'missing_required_terms': [],
  'overlap_terms': ['00m',
                    '152',
                    '154',
                    '32m',
                    '50m',
                    's',
                    '京密',
                    '京密引',
                    '京密引水',
                    '其中',
                    '包含',
                    '包含京',
                    '包含京密',
                    '含京',
                    '含京密',
                    '含京密引',
                    '密引',
                    '密引水',
                    '密引水渠',
                    '常调'],
  'reason': 'claim 和引用证据有 61 个关键词重合',
  'status': 'overlap_supported',
  'text': '**常规控泄流量包含引水流量**：在正常调度规程中，水库的控泄流量是包含京密引水渠引水流量的。例如，当水库水位在152.00m至154.32m之间时，

## 7. 构造一个故意错误的 claim

为了看验证器是否有区分能力，我们加一条明显没有证据支持的声明：

```text
密云水库防御洪水方案要求每季度组织一次全员演练。[T1]
```

这条 claim 有引用，但引用不支持它。


In [6]:
bad_claim = {
    'claim_id': 'C_BAD',
    'text': '密云水库防御洪水方案要求每季度组织一次全员演练。',
    'citations': ['T1' if 'T1' in citation_map else sorted(citation_map)[0]],
}

bad_rule_result = rule_check_claim(bad_claim, citation_map)
pprint(bad_rule_result)

{'citations': ['T1'],
 'claim_id': 'C_BAD',
 'missing_required_terms': ['全员', '每季度', '演练'],
 'overlap_terms': [],
 'reason': "claim 含有精确约束词，但引用证据未出现: ['全员', '每季度', '演练']",
 'status': 'missing_required_terms',
 'text': '密云水库防御洪水方案要求每季度组织一次全员演练。'}


## 8. 小模型版 groundedness 评审

规则检查只能看关键词重合，不能真正理解“是否被支持”。

所以可以再加一个小模型评审器，让模型只做三分类：

- `supported`：证据直接支持。
- `unsupported`：证据不支持或相反。
- `unclear`：证据相关但不足以推出。

注意：评审模型也可能错，所以它是验证链的一环，不是绝对真理。


In [7]:
RAG_CONFIG = {
    'model_base_url': os.getenv('RAG_MODEL_BASE_URL', 'http://192.168.102.19:8082/v1'),
    'judge_model': os.getenv('RAG_JUDGE_MODEL', 'qwen2.5-0.5b-instruct'),
}

RUN_JUDGE_MODEL = True


def build_judge_prompt(claim: dict, citation_map: dict) -> list[dict]:
    evidence_lines = []
    for citation in claim.get('citations', []):
        item = citation_map.get(citation)
        if item is None:
            continue
        evidence_lines.append(f'[{citation}] {item["evidence_text"][:900]}')

    user_prompt = f'''请判断 claim 是否被 evidence 支持。

claim:
{claim['text']}

引用证据:
{chr(10).join(evidence_lines)}

只输出 JSON，不要输出其他文字。格式：
{{"label":"supported|unsupported|unclear","reason":"一句话原因"}}'''

    return [
        {
            'role': 'system',
            'content': '你是事实一致性评审器，只判断 claim 是否能从给定 evidence 推出。',
        },
        {'role': 'user', 'content': user_prompt},
    ]


def parse_json_object(text: str) -> dict:
    match = re.search(r'\{.*\}', text, flags=re.S)
    if not match:
        return {'label': 'unclear', 'reason': f'模型没有返回 JSON: {text[:120]}'}
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        return {'label': 'unclear', 'reason': f'JSON 解析失败: {exc}'}
    label = data.get('label')
    if label not in {'supported', 'unsupported', 'unclear'}:
        label = 'unclear'
    return {'label': label, 'reason': str(data.get('reason', '')).strip()}


def judge_claim_with_model(client: OpenAI, claim: dict, citation_map: dict) -> dict:
    if not claim.get('citations'):
        return {'label': 'unsupported', 'reason': 'claim 没有引用证据'}
    messages = build_judge_prompt(claim, citation_map)
    response = client.chat.completions.create(
        model=RAG_CONFIG['judge_model'],
        messages=messages,
        temperature=0,
        max_tokens=180,
    )
    raw = response.choices[0].message.content or ''
    parsed = parse_json_object(raw)
    return {**parsed, 'raw': raw}

sample_claims_for_judge = claims[:4] + [bad_claim]

if RUN_JUDGE_MODEL:
    judge_client = OpenAI(api_key=os.getenv('RAG_MODEL_API_KEY', 'EMPTY'), base_url=RAG_CONFIG['model_base_url'])
    judge_results = []
    for claim in sample_claims_for_judge:
        try:
            judgement = judge_claim_with_model(judge_client, claim, citation_map)
        except Exception as exc:
            judgement = {'label': 'unclear', 'reason': f'模型评审失败: {exc}', 'raw': ''}
        judge_results.append({**claim, **judgement})
else:
    judge_results = [{**claim, 'label': 'unclear', 'reason': 'RUN_JUDGE_MODEL=False', 'raw': ''} for claim in sample_claims_for_judge]

pprint(judge_results)

[{'citations': [],
  'claim_id': 'C1',
  'label': 'unsupported',
  'reason': 'claim 没有引用证据',
  'text': '根据提供的证据，关于“密引水渠引水流量不能满足泄洪要求”的处理方式，证据中并未直接给出针对该特定情况的独立处理措施，但提供了相关的调度原则和超标准洪水下的应急方案。以下是基于证据的要点'},
 {'citations': ['T2'],
  'claim_id': 'C2',
  'label': 'supported',
  'raw': '{"label":"supported","reason":"evidence支持claim"}',
  'reason': 'evidence支持claim',
  'text': '**常规控泄流量包含引水流量**：在正常调度规程中，水库的控泄流量是包含京密引水渠引水流量的。例如，当水库水位在152.00m至154.32m之间时，控泄流量为600m³/s，其中已包含京密引水渠引水流量50m³/s '
          '。'},
 {'citations': ['T2'],
  'claim_id': 'C3',
  'label': 'supported',
  'raw': '{"label":"supported","reason":"evidence支持该说法"}',
  'reason': 'evidence支持该说法',
  'text': '**高水位时全开闸门不控泄**：当水库水位超过157.50m时，输、泄水建筑物闸门全部开启，不再进行控泄，此时引水流量对泄洪能力的限制不再适用 '
          '。'},
 {'citations': ['T1'],
  'claim_id': 'C4',
  'label': 'supported',
  'raw': '{"label":"supported","reason":"证据支持了超标准洪水下的极端措施"}',
  'reason': '证据支持了超标准洪水下的极端措施',
  'text': '**超标准洪水下的极端措施**：若发生超标准洪水，且敞开各输、泄水建筑物泄洪仍不能满足工程安全要求时，经北京市水务局批准后，可选择炸开副坝的形式进行泄

## 9. 汇总报告

真实系统里，groundedness 通常不会只给一个布尔值。

更有用的是输出一个报告：

- 哪些 claim 缺引用。
- 哪些 claim 引用弱。
- 哪些 claim 被评审模型判为 unsupported / unclear。
- 是否允许直接返回给用户。


In [8]:
def summarize_groundedness(rule_results: list[dict], judge_results: list[dict]) -> dict:
    rule_counts = {}
    for item in rule_results:
        rule_counts[item['status']] = rule_counts.get(item['status'], 0) + 1

    judge_counts = {}
    for item in judge_results:
        judge_counts[item['label']] = judge_counts.get(item['label'], 0) + 1

    hard_fail_statuses = {'missing_citation', 'unknown_citation', 'missing_required_terms'}
    has_hard_rule_fail = any(item['status'] in hard_fail_statuses for item in rule_results)
    has_unsupported_judge = any(item['label'] == 'unsupported' for item in judge_results)

    return {
        'claim_count': len(rule_results),
        'rule_counts': rule_counts,
        'judge_counts': judge_counts,
        'has_hard_rule_fail': has_hard_rule_fail,
        'has_unsupported_judge': has_unsupported_judge,
        'safe_to_return_without_review': not has_hard_rule_fail and not has_unsupported_judge,
    }

summary = summarize_groundedness(rule_results, judge_results)
pprint(summary)

{'claim_count': 7,
 'has_hard_rule_fail': True,
 'has_unsupported_judge': True,
 'judge_counts': {'supported': 4, 'unsupported': 1},
 'rule_counts': {'missing_citation': 1, 'overlap_supported': 6},
 'safe_to_return_without_review': False}


## 10. 保存 groundedness 报告

保存报告的意义是方便后续排查：

- 是召回证据不够？
- 是回答模型过度发挥？
- 是引用没挂到正确证据？
- 是评审模型误判？

这些问题不能只靠最终答案判断，必须保留中间轨迹。


In [9]:
groundedness_report = {
    'question': qa_result['question'],
    'answer_source': qa_result.get('answer_source'),
    'summary': summary,
    'rule_results': rule_results,
    'judge_results': judge_results,
    'bad_claim_demo': bad_rule_result,
    'qa_result_path': str(qa_result_path),
    'evidence_package_path': str(evidence_package_path),
}

groundedness_report_path.write_text(
    json.dumps(groundedness_report, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('groundedness_report_path:', groundedness_report_path)
print('size KB:', round(groundedness_report_path.stat().st_size / 1024, 2))

groundedness_report_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/groundedness_report.json
size KB: 9.16


## 11. 本课小结

这一课的核心结论：

1. 引用验证只能证明“引用存在”，不能证明“事实被支持”。
2. groundedness check 的粒度应该是 claim，不是整篇回答。
3. 规则验证适合做低成本第一道门槛。
4. 模型评审适合补语义判断，但不能当作绝对真理。
5. 生产系统要保留报告，用于定位问题属于召回、回答、引用还是评审。

到这里，RAG / GraphRAG 的教学闭环已经从“能回答”推进到了“能解释回答是否可靠”。
